In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score

print("1. โหลดข้อมูล...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
target_col = 'History of HeartDisease or Attack'

train = train.dropna(subset=[target_col])
X = train.drop(columns=['ID', target_col])
y = train[target_col]
X_test = test.drop(columns=['ID'])

# แปลง Target (Yes/No) เป็น 1/0
y_le = LabelEncoder()
y_encoded = y_le.fit_transform(y) # No=0, Yes=1

# คำนวณสัดส่วนคลาสอัตโนมัติ (ช่วยให้โมเดลให้ความสำคัญกับคนเป็นโรค)
scale_pos_weight = sum(y_encoded == 0) / sum(y_encoded == 1)

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns
X_all = pd.concat([X, X_test], axis=0)

print("2. จัดการ Missing Value และแปลงข้อมูลแบบ One-Hot Encoding...")
imputer_num = SimpleImputer(strategy='median')
X_all[num_cols] = imputer_num.fit_transform(X_all[num_cols])

imputer_cat = SimpleImputer(strategy='most_frequent')
X_all[cat_cols] = imputer_cat.fit_transform(X_all[cat_cols])

# ⭐️ ใช้ One-Hot แยกคอลัมน์ ดีกว่าตัวเลขเรียงลำดับสำหรับ XGBoost
X_all = pd.get_dummies(X_all, columns=cat_cols, drop_first=True)

X_train_full = X_all.iloc[:len(X)]
X_test_imputed = X_all.iloc[len(X):]

print("3. แบ่งข้อมูล Validation เพื่อจำลองหา Threshold ที่ดีที่สุด...")
# แบ่งข้อมูล 20% เอาไว้ทดสอบหาสัดส่วนที่ F2 สูงสุด
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

print("4. กำลังเทรนโมเดล XGBoost (ใช้เวลาสักครู่)...")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, # บังคับให้เน้นทาย Yes
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_tr, y_tr)

print("5. ค้นหา Threshold ที่ปั่นคะแนน F2-Score ได้สูงสุด...")
val_probs = xgb.predict_proba(X_val)[:, 1]
best_threshold = 0.5
best_f2 = 0

# วนลูปทดสอบ Threshold ตั้งแต่ 0.10 ถึง 0.90
thresholds = np.arange(0.1, 0.9, 0.05)
for thresh in thresholds:
    val_preds = (val_probs >= thresh).astype(int)
    f2 = fbeta_score(y_val, val_preds, beta=2) # วัดผลด้วย F2-Score ล้วนๆ
    if f2 > best_f2:
        best_f2 = f2
        best_threshold = thresh

print(f"   => พบ Threshold อัจฉริยะที่: {best_threshold:.2f} (จำลอง F2-Score ได้: {best_f2:.5f})")

print("6. เทรนโมเดลซ้ำด้วยข้อมูล 100% และทำนายผล Test...")
# เทรนใหม่แบบจัดเต็ม
xgb.fit(X_train_full, y_encoded)
test_probs = xgb.predict_proba(X_test_imputed)[:, 1]

# ⭐️ ใช้ Threshold ที่เพิ่งหามาได้
final_preds = (test_probs >= best_threshold).astype(int)
preds_labels = y_le.inverse_transform(final_preds)

submission = pd.DataFrame({'ID': test['ID'], target_col: preds_labels})
submission.to_csv('heart_disease_xgboost_f2.csv', index=False)
print("✅ เสร็จสมบูรณ์! ดาวน์โหลดไฟล์ 'heart_disease_xgboost_f2.csv' ไป Submit ได้เลยครับ")

1. โหลดข้อมูล...
2. จัดการ Missing Value และแปลงข้อมูลแบบ One-Hot Encoding...
3. แบ่งข้อมูล Validation เพื่อจำลองหา Threshold ที่ดีที่สุด...
4. กำลังเทรนโมเดล XGBoost (ใช้เวลาสักครู่)...
5. ค้นหา Threshold ที่ปั่นคะแนน F2-Score ได้สูงสุด...
   => พบ Threshold อัจฉริยะที่: 0.55 (จำลอง F2-Score ได้: 0.53595)
6. เทรนโมเดลซ้ำด้วยข้อมูล 100% และทำนายผล Test...
✅ เสร็จสมบูรณ์! ดาวน์โหลดไฟล์ 'heart_disease_xgboost_f2.csv' ไป Submit ได้เลยครับ
